# 2015 SOXX 原始披露来源

系列专属 accession、五个报告日与预算见 `plan.json`。不把过渡年强制变为四季。


In [1]:
import json,hashlib,subprocess,tempfile,tarfile,shlex,time
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/stage1_soxx_2015_transition_concentration_source_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['source_series_calendar_sha256']==sha(root/'research/stage1_soxx_series_filing_search_20261023/series_report_calendar_inventory.json')
rawdir=root/'data/snapshots/stage1_soxx_2015_transition_concentration_source_20261023';rawdir.mkdir(parents=True,exist_ok=True)
claim=card/'attempt_started.json';out=card/'transport_result.json'
assert not claim.exists() and not out.exists() and not any(rawdir.iterdir())
claim.write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'started_at_utc':datetime.now(timezone.utc).isoformat(),'maximum_GETs':5},indent=2)+'\n')
requests=[]
for j,item in enumerate(plan['reports']):
 if j:time.sleep(plan['request']['minimum_seconds_between_requests'])
 raw=rawdir/(item['accession']+'_'+item['primary_document']);headers=rawdir/(raw.name+'.headers.txt')
 started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--compressed','--http1.1','--connect-timeout',str(plan['request']['connect_timeout_seconds']),
  '--max-time',str(plan['request']['total_timeout_seconds_each']),'--max-filesize',str(plan['request']['max_response_bytes_each']),
  '--fail-with-body','-sS','-D',str(headers),'-o',str(raw),'-w','%{http_code}\n%{url_effective}',
  '-H','User-Agent: '+plan['request']['user_agent'],'-H','Accept: text/html',item['url']]
 proc=subprocess.run(cmd,capture_output=True,text=True,timeout=plan['request']['total_timeout_seconds_each']+15)
 ended=datetime.now(timezone.utc).isoformat();lines=proc.stdout.splitlines();status=int(lines[0]) if lines and lines[0].isdigit() else None
 entry={'report_date':item['report_date'],'form':item['form'],'accession':item['accession'],'accepted_at_utc':item['accepted_at_utc'],
  'url':item['url'],'requested_at_utc':started,'received_at_utc':ended,'http_status':status,
  'final_url':lines[1] if len(lines)>1 else None,'curl_exit_code':proc.returncode,'curl_stderr':proc.stderr[:400],
  'raw_path':str(raw.relative_to(root)),'raw_sha256':sha(raw) if raw.exists() else None,'raw_bytes':raw.stat().st_size if raw.exists() else 0,
  'headers_sha256':sha(headers) if headers.exists() else None,'headers_bytes':headers.stat().st_size if headers.exists() else 0}
 requests.append(entry)
 if proc.returncode or status!=200 or entry['final_url']!=item['url'] or not raw.exists() or raw.stat().st_size>plan['request']['max_response_bytes_each']:
  entry['stop_reason']='transport_or_identity_failed';break
receipt={'status':'five_original_2015_SEC_bodies_received_identity_pending' if len(requests)==5 and all('stop_reason' not in x for x in requests) else 'original_2015_body_batch_stopped',
 'plan_sha256':sha(card/'plan.json'),'GETs_consumed':len(requests),'requests':requests,'SOXX_schedule_identities_qualified':0,
 'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
out.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='soxx-2015-original-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for path in [claim,out,*sorted(rawdir.iterdir())]:
   if path.exists():tar.add(path,arcname=path.name)
 settings=_settings(root);digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='soxx-2015-original-restore-') as tmp2:
  copy=Path(tmp2)/'restore.tar.gz'
  with copy.open('wb') as out_file:
   check=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],
    'cat '+shlex.quote(remote)],stdout=out_file,stderr=subprocess.PIPE,timeout=120)
  assert check.returncode==0 and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as stored:
   for entry in requests:
    if entry['raw_sha256'] is None:continue
    member=stored.extractfile(Path(entry['raw_path']).name)
    assert member and hashlib.sha256(member.read()).hexdigest()==entry['raw_sha256']
 receipt['nas_evidence']=evidence;receipt['nas_scoped_restore_verified']=True
 out.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
print({'status':receipt['status'],'GETs':len(requests),'raw_MB':round(sum(x['raw_bytes'] for x in requests)/1e6,2),'NAS_restore':True})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'original_2015_body_batch_stopped', 'GETs': 4, 'raw_MB': 138.7, 'NAS_restore': True}


In [1]:
import json,hashlib,subprocess,tempfile,tarfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/stage1_soxx_2015_transition_concentration_source_20261023';plan=json.loads((card/'2015_remaining_size_repair_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['parent_plan_sha256']==sha(card/'plan.json')
assert plan['stopped_transport_result_sha256']==sha(card/'transport_result.json')
prior=json.loads((card/'transport_result.json').read_text());assert prior['GETs_consumed']==4 and prior['nas_scoped_restore_verified']
assert prior['nas_evidence']['sha256']==plan['existing_source_NAS_archive_sha256']
for x in plan['reuse_existing']:
 raw=root/x['raw_path'];assert raw.stat().st_size==x['raw_bytes'] and sha(raw)==x['raw_sha256']
rawdir=root/'data/snapshots/stage1_soxx_2015_transition_concentration_source_20261023';item=plan['new_exact_candidate']
raw=rawdir/(item['accession']+'_'+item['primary_document']);headers=rawdir/(raw.name+'.headers.txt')
claim=card/'remaining_attempt_started.json';out=card/'remaining_transport_result.json'
assert not claim.exists() and not out.exists() and not raw.exists() and not headers.exists()
started=datetime.now(timezone.utc).isoformat();claim.write_text(json.dumps({'plan_sha256':sha(card/'2015_remaining_size_repair_plan.json'),'started_at_utc':started,'new_GET_attempts_started':1},indent=2)+'\n')
cmd=['curl','--compressed','--http1.1','--connect-timeout',str(plan['new_request']['connect_timeout_seconds']),
 '--max-time',str(plan['new_request']['total_timeout_seconds']),'--max-filesize',str(plan['new_request']['max_response_bytes']),
 '--fail-with-body','-sS','-D',str(headers),'-o',str(raw),'-w','%{http_code}\n%{url_effective}',
 '-H','User-Agent: '+plan['new_request']['user_agent'],'-H','Accept: text/html',item['url']]
proc=subprocess.run(cmd,capture_output=True,text=True,timeout=plan['new_request']['total_timeout_seconds']+15)
ended=datetime.now(timezone.utc).isoformat();lines=proc.stdout.splitlines();status=int(lines[0]) if lines and lines[0].isdigit() else None
receipt={'status':'remaining_2015_Dec_body_transport_unqualified','plan_sha256':sha(card/'2015_remaining_size_repair_plan.json'),
 'report_date':item['report_date'],'accession':item['accession'],'url':item['url'],'request_started_at_utc':started,'response_finished_at_utc':ended,
 'http_status':status,'final_url':lines[1] if len(lines)>1 else None,'curl_exit_code':proc.returncode,'curl_stderr':proc.stderr[:400],
 'raw_path':str(raw.relative_to(root)),'raw_sha256':sha(raw) if raw.exists() else None,'raw_bytes':raw.stat().st_size if raw.exists() else 0,
 'headers_sha256':sha(headers) if headers.exists() else None,'headers_bytes':headers.stat().st_size if headers.exists() else 0,
 'new_GETs':1,'repeated_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
if proc.returncode==0 and status==200 and receipt['final_url']==item['url'] and raw.exists() and raw.stat().st_size<=plan['new_request']['max_response_bytes']:
 receipt['status']='exact_2015_Dec_original_body_received_identity_pending'
out.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='soxx-2015-dec-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for path in (raw,headers,claim,out):
   if path.exists():tar.add(path,arcname=path.name)
 settings=_settings(root);digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='soxx-2015-dec-restore-') as tmp2:
  copy=Path(tmp2)/'restore.tar.gz'
  with copy.open('wb') as out_file:
   check=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],
    'cat '+shlex.quote(remote)],stdout=out_file,stderr=subprocess.PIPE,timeout=90)
  assert check.returncode==0 and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as stored:
   member=stored.extractfile(raw.name) if raw.exists() else None
   if member is not None:assert hashlib.sha256(member.read()).hexdigest()==receipt['raw_sha256']
 receipt['nas_evidence']=evidence;receipt['nas_scoped_restore_verified']=bool(raw.exists())
 out.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
print({'status':receipt['status'],'bytes':receipt['raw_bytes'],'NAS_restore':receipt['nas_scoped_restore_verified']})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'exact_2015_Dec_original_body_received_identity_pending', 'bytes': 38084849, 'NAS_restore': True}


## 中文结论

执行后见 `summary.json` 与 `conclusion.md`；来源闭合不是 SOXX 超额。
